#### Data Balancing (Not Applicable)

**Decision: Not Applicable.**
Because the model objectives (Tasks A and B) predict continuous values (Watts and kWh), this is a **regression problem, not classification**.

> Balancing methods (like SMOTE or class weights) address class imbalance in classification. Since both targets here are continuous, there are no classes to balance, meaning there is zero resampling-related leakage risk.

The analogous issue for regression is a skewed target, which we handle instead via robust evaluation metrics (MAE, MedAE) and log-target experiments later in the pipeline.

##### Shared Helpers
Instead of resampling, we use this stage to define **Shared Helpers**—reusable pipeline components and metric functions that will power the baseline and advanced models.

In [2]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from xgboost import XGBRegressor
from sklearn.metrics import (mean_absolute_error, mean_squared_error,r2_score, mean_absolute_percentage_error)

CAT = ['Service_Class', 'Network_Slice', 'Network_Function', 'Deployment_Zone', 'Traffic_Direction']
ENERGY_DERIVED = ['Energy_Consumption_kWh', 'Energy_Efficiency_Mbps_per_kWh', 'Carbon_Emission_kg', 'Operational_Cost_USD']
CONG = ['Network_Condition', 'Congestion_Index', 'QoS_Satisfaction_Score', 'SLA_Violation_Count']
FUTURE = ['Future_Traffic_Demand_Mbps', 'Future_Congestion_Index', 'Predicted_Energy_Requirement_kWh', 'Energy_Management_Status']

DROP_BASE = ['Record_ID', 'Timestamp', 'Month', 'split', 'P_struct', 'Uplink_Traffic_Mbps', 'Downlink_Traffic_Mbps', 'Total_Power_Watt'] + ENERGY_DERIVED + CONG + FUTURE

def make_preprocessor(num_cols, cat_cols, scale=False, skewed=()):
    """Builds a robust ColumnTransformer for numerical and categorical data."""
    skewed_list = [c for c in num_cols if c in set(skewed)]
    plain_list = [c for c in num_cols if c not in skewed]
    tf = []
    
    if scale:
        if plain_list:
            tf.append(('num', Pipeline([('imp', SimpleImputer(strategy='median')), 
                                        ('sc', StandardScaler())]), plain_list))
        if skewed_list:
            tf.append(('skewed', Pipeline([('imp', SimpleImputer(strategy='median')), 
                                           ('log', FunctionTransformer(np.log1p, feature_names_out='one-to-one')), 
                                           ('sc', StandardScaler())]), skewed_list))
    else:
        tf.append(('num', SimpleImputer(strategy='median'), list(num_cols)))
        
    if cat_cols:
        tf.append(('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), list(cat_cols)))
        
    return ColumnTransformer(tf)

def reg_pipe(est, num, scale=False, skewed=()):
    """Wraps the preprocessor and estimator into a single Pipeline."""
    return Pipeline([('prep', make_preprocessor(num, CAT, scale, skewed)), ('reg', est)])

def xgb_reg(**kw):
    """Returns an XGBRegressor with standardized default hyperparameters."""
    p = dict(n_estimators=250, max_depth=3, learning_rate=0.05, subsample=0.8,colsample_bytree=0.8, min_child_weight=5, tree_method='hist', n_jobs=1, random_state=42, verbosity=0)
    p.update(kw)
    return XGBRegressor(**p)

def reg_metrics(y, p):
    """Calculates regression metrics (MAE, MSE, RMSE, RMSE_Norm_%, R2, MedAE, MAPE_%) for evaluation."""
    y, p = np.asarray(y, float), np.asarray(p, float)
    
    mse = mean_squared_error(y, p)
    rmse = mse ** 0.5
    
    d = {
        'MAE': mean_absolute_error(y, p),
        'MSE': mse,
        'RMSE': rmse,
        'RMSE_Norm_%': (rmse / y.mean()) * 100,
        'R2': r2_score(y, p),
        'MedAE': np.median(np.abs(y - p)),
        'MAPE_%': mean_absolute_percentage_error(y, p) * 100
    }
    return d

print("Shared helpers defined successfully.")

Shared helpers defined successfully.
